# NB2: CD/CI Strategy Meta-learner

데이터셋 메타피처 + 모델 구성 (arch, scope, level) → **CD vs CI 학습전략 우열 점수** (perf / time / mem) 예측.

**점수 공식** (`analysis/results_viz.ipynb` 의 heatmap 공식과 동일):

```
score = (X_CD - X_CI) / max(X_CD, X_CI)   ∈  [-1, +1]
  +1  : CI 가 매우 좋음
  -1  : CD 가 매우 좋음
   0  : 동률
```

X 는 `mse`, `train_time_mean`, `num_params` 세 가지에 각각 적용 (multi-output).
mse 는 `compute_perf_score`, time/mem 은 `compute_efficiency_score` 로 의미 분리.

**입력**: NB1 출력 + `../results_BrickTS/**/*.json` + `../results/**/*.json`

**핵심 출력** (`outputs/nb2/`):
- `brickts_long.csv` / `brickts_wide.csv` — 원본 long + (dataset, arch, scope, level) wide
- `benchmark_long.csv` / `benchmark_wide_per_model.csv` / `benchmark_wide_bucket.csv`
- `ml_metrics_per_target.csv` / `ml_predictions.csv`
- `feature_importance_tree.csv`
- `best_model.pkl` / `best_model_config.csv`
- `brickts_vs_benchmark.csv` (§10 raw 비교표)

**파이프라인**:
1. BrickTS 결과 로딩 (long, 원본 보존) → wide reshape → 3 score 계산
2. 외부 벤치마크 로딩 → per-model score → bucket median
3. Feature subset → LODO multi-output 학습 (LightGBM + XGBoost 둘 다)
4. auc(perf_score) 기준 best 모델 자동 선정 + LODO_DATASETS 전체 재학습
5. §10 BrickTS vs 외부 raw 비교 + 2-class(CD/CI) confusion matrix (sanity check)
6. 추론 인터페이스는 NB3 §3 으로 이동


## Section 0 — Setup + NB1 출력 로드

In [ ]:
%pip install --quiet numpy pandas scipy scikit-learn statsmodels pycatch22 antropy nolds PyWavelets EntropyHub xgboost lightgbm matplotlib seaborn tqdm

In [ ]:
from __future__ import annotations
import glob, json, re, pickle, warnings
from pathlib import Path
from typing import Optional
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

np.random.seed(42)

NB1 = Path('outputs/nb1')
OUT = Path('outputs/nb2'); OUT.mkdir(parents=True, exist_ok=True)

dataset_meta   = pd.read_csv(NB1 / 'dataset_metadata.csv').set_index('dataset')
feature_groups = pd.read_csv(NB1 / 'feature_groups.csv')
meta_full      = pd.read_csv(NB1 / 'meta_features_full.csv').set_index('dataset')

print(f'meta_full: {meta_full.shape}  |  feature_groups: {feature_groups.shape}')
dataset_meta


## Section 1 — BrickTS 결과 로딩 (long format, 원본 보존)

OT 채널의 `mse`, `mae`, `num_params`, `train_time_mean` 4가지를 추출. CD/CI 별 row 그대로 유지하여 원본을 손상하지 않는다.


In [ ]:
ARCH_TYPES  = ['mlp', 'rnn', 'cnn', 'transformer']
SCOPE_TYPES = ['global', 'local', 'hierarchical', 'sparse']
LEVEL_TYPES = ['direct', 'decomposition', 'spectral']
STRATEGIES  = ['CD', 'CI']

# 표준 벤치마크 폴더만 수동 매핑. Milano-<K> 는 results_BrickTS 스캔으로 자동 등록 (register_milano_folders).
FOLDER_TO_NAME = {
    'ETTh1': 'ETTh1', 'ETTh2': 'ETTh2', 'ETTm1': 'ETTm1', 'ETTm2': 'ETTm2',
    'traffic': 'Traffic', 'electricity': 'Electricity',
    'exchange_rate': 'Exchange', 'exchange': 'Exchange',
    'weather': 'Weather', 'national_illness': 'ILI', 'ILI': 'ILI',
}

def _folder_to_name(folder: str) -> str:
    """폴더명 → 데이터셋 표준명. milano_<K> / milano-<K> 는 자동으로 Milano-<K>."""
    if folder in FOLDER_TO_NAME:
        return FOLDER_TO_NAME[folder]
    m = re.match(r'^milano[_-](\d+)$', folder, re.I)
    if m:
        return f'Milano-{m.group(1)}'
    return folder


RESULTS_BRICKTS_CANDIDATES = ['../results_BrickTS', 'results_BrickTS', './results_BrickTS']


def register_milano_folders(results_dir_candidates) -> list:
    """results_BrickTS 하위 폴더를 스캔해 모든 milano_<K> 를 FOLDER_TO_NAME 에 자동 등록."""
    found = []
    for base in results_dir_candidates:
        bp = Path(base)
        if not bp.is_dir():
            continue
        for sub in bp.iterdir():
            if not sub.is_dir():
                continue
            mm = re.match(r'^milano[_-](\d+)$', sub.name, re.I)
            if mm:
                std = f'Milano-{mm.group(1)}'
                FOLDER_TO_NAME[sub.name] = std
                FOLDER_TO_NAME[std] = std
                found.append(std)
    return sorted(set(found), key=lambda s: int(s.split('-')[1]))


_milano_registered = register_milano_folders(RESULTS_BRICKTS_CANDIDATES)
print(f'Auto-registered {len(_milano_registered)} Milano folders to FOLDER_TO_NAME: {_milano_registered}')


# scalability dict 에서 train_time 추출 (BrickTS는 'train_time', 일반 벤치마크는 'train_time_mean' 사용)
TRAIN_TIME_KEYS = ('train_time_mean', 'train_time', 'training_time', 'training_time_seconds')


def _scal_train_time(scal: dict) -> float:
    for k in TRAIN_TIME_KEYS:
        v = scal.get(k)
        if v is not None:
            return float(v)
    return float('nan')


def _ot_metrics_from_payload(data: dict, mode: str, ot_name: str) -> tuple:
    """OT 채널의 (mse, mae, num_params, train_time) 추출. 키 없으면 NaN."""
    ch = data.get('per_channel', {}).get(ot_name) if mode == 'CD' else data.get(ot_name)
    if not ch:
        return (np.nan, np.nan, np.nan, np.nan)
    scal = (data.get('scalability') if mode == 'CD' else ch.get('scalability')) or {}
    return (float(ch.get('mse', np.nan)),
            float(ch.get('mae', np.nan)),
            float(scal.get('num_params', np.nan)),
            _scal_train_time(scal))


def _parse_brickts_path(path: Path) -> Optional[dict]:
    parts = path.parts
    if 'results_BrickTS' not in parts:
        return None
    ri = parts.index('results_BrickTS')
    if ri + 4 >= len(parts):
        return None
    combo = parts[ri + 4]
    stem = path.stem.replace('_results', '')
    mode = None
    if stem.startswith(combo + '_'):
        rest = stem[len(combo) + 1:].split('_')
        if rest and rest[0] in STRATEGIES:
            mode = rest[0]
    if mode is None:
        _, tail = stem.rsplit('_', 1)
        mode = tail if tail in STRATEGIES else None
    if mode is None:
        return None
    axis = combo.split('_')
    if len(axis) < 3 or axis[0] not in ARCH_TYPES or axis[1] not in SCOPE_TYPES:
        return None
    level = '_'.join(axis[2:])
    if level not in LEVEL_TYPES:
        return None
    try:
        seq_len, pred_len = map(int, parts[ri + 2].split('_'))
    except ValueError:
        seq_len = pred_len = -1
    return {
        'dataset': _folder_to_name(parts[ri + 1]),
        'arch': axis[0], 'scope': axis[1], 'level': level,
        'strategy': mode, 'seq_len': seq_len, 'pred_len': pred_len,
    }


def load_brickts_ot_results(results_dir: str, ot_name_by_dataset: dict) -> pd.DataFrame:
    files = glob.glob(f'{results_dir}/**/*_results.json', recursive=True)
    print(f'Found {len(files)} JSON files under {results_dir}')
    rows = []
    for fp in files:
        meta = _parse_brickts_path(Path(fp))
        if meta is None:
            continue
        with open(fp) as f:
            data = json.load(f)
        ot_name = ot_name_by_dataset.get(meta['dataset'], 'OT')
        mse, mae, n_params, t = _ot_metrics_from_payload(data, meta['strategy'], ot_name)
        rows.append({**meta, 'mse': mse, 'mae': mae,
                     'num_params': n_params, 'train_time': t})
    return pd.DataFrame(rows)


ot_map = dataset_meta['ot_name'].to_dict()
brickts_long = pd.DataFrame()
for cand in RESULTS_BRICKTS_CANDIDATES:
    df = load_brickts_ot_results(cand, ot_map)
    if len(df):
        brickts_long = df
        print(f'Using results_dir = {cand}')
        break

brickts_long.to_csv(OUT / 'brickts_long.csv', index=False)
print('\nLoaded rows:', len(brickts_long))
print('\nPer-dataset combo coverage (target = 96 = 48 combos x 2 strategies):')
print(brickts_long.groupby('dataset').size().sort_index())
print('\nNaN count per metric:')
print(brickts_long[['mse', 'mae', 'num_params', 'train_time']].isna().sum())
brickts_long.head()

## Section 2 — Wide reshape + 점수 계산

(dataset, arch, scope, level) 단위로 reshape (CD/CI 컬럼화) → 3개 score 계산.

`score = (X_CD - X_CI) / max(X_CD, X_CI)` ∈ [-1, +1]
- **+1** : CI 우월 / **-1** : CD 우월 / **0** : 동률


In [ ]:
def _signed_diff_core(cd: pd.Series, ci: pd.Series) -> pd.Series:
    """(cd - ci) / max(cd, ci) ∈ [-1, +1] — 공통 핵심.

    +1: CI 가 매우 좋음 (CI 의 값이 작을수록 — mse / time / num_params 모두 작을수록 좋음)
    -1: CD 가 매우 좋음
     0: 동률
    NaN / Inf 입력 또는 max == 0 → NaN 출력 (robust).
    """
    denom = np.maximum(cd.values, ci.values)
    out = np.full_like(denom, np.nan, dtype=float)
    mask = (denom > 0) & np.isfinite(cd) & np.isfinite(ci)
    out[mask] = (cd.values[mask] - ci.values[mask]) / denom[mask]
    return pd.Series(out, index=cd.index)


def compute_perf_score(mse_cd: pd.Series, mse_ci: pd.Series) -> pd.Series:
    """MSE 기반 성능 비교 score.

    results_viz.ipynb 의 `create_performance_heatmap` 공식 그대로:
      score = (cd - ci) / max(cd, ci),  양수: CI better,  음수: CD better.

    원본 함수는 perf_diff_pct = score * 100 (% 단위 표기). 학습에는 [-1, +1] 사용,
    시각화 시 % 변환 가능.
    """
    return _signed_diff_core(mse_cd, mse_ci)


def compute_efficiency_score(cost_cd: pd.Series, cost_ci: pd.Series) -> pd.Series:
    """Time / Memory (num_params 등 cost proxy) 기반 효율성 비교 score.

    results_viz.ipynb 의 `create_efficiency_comparison_heatmap` 공식 그대로:
      sign · (1 - min/max) ≡ (cd - ci) / max(cd, ci),  양수: CI more efficient.

    원본 함수의 robust 처리 (NaN/Inf skip + big == 0 처리) 를 _signed_diff_core 가
    이미 포함하므로 동일 결과. annot 시각화 시 big/small 배수로 표기 권장.
    """
    return _signed_diff_core(cost_cd, cost_ci)


# 하위 호환을 위해 signed_diff 도 유지 (compute_perf_score 와 같은 함수)
signed_diff = compute_perf_score


REQUIRED_WIDE_COLS = ['mse_CD', 'mse_CI',
                       'train_time_CD', 'train_time_CI',
                       'num_params_CD', 'num_params_CI']


def to_wide_with_scores(long_df: pd.DataFrame) -> pd.DataFrame:
    wide = long_df.pivot_table(
        index=['dataset', 'arch', 'scope', 'level'],
        columns='strategy',
        values=['mse', 'train_time', 'num_params'],
    ).reset_index()
    wide.columns = [f'{a}_{b}' if b else a for a, b in wide.columns]
    # pivot_table은 모든 값이 NaN인 그룹의 컬럼을 생성하지 않으므로 누락 컬럼 NaN으로 강제 생성
    for col in REQUIRED_WIDE_COLS:
        if col not in wide.columns:
            wide[col] = np.nan
    wide = wide.dropna(subset=['mse_CD', 'mse_CI'])  # perf score 정의 가능 조건

    # MSE (성능) 와 cost (효율성) 는 의미가 달라 서로 다른 함수 호출.
    # 수식 자체는 동일하지만 사용자가 코드를 읽을 때 의도가 분명해짐.
    wide['perf_score'] = compute_perf_score(wide['mse_CD'], wide['mse_CI'])
    wide['time_score'] = compute_efficiency_score(wide['train_time_CD'], wide['train_time_CI'])
    wide['mem_score']  = compute_efficiency_score(wide['num_params_CD'], wide['num_params_CI'])
    return wide


brickts_wide = to_wide_with_scores(brickts_long)
brickts_wide.to_csv(OUT / 'brickts_wide.csv', index=False)
print(f'brickts_wide shape: {brickts_wide.shape} (expected ~480 rows = 10 datasets x 48 combos)')
print('\nNaN count per score (NaN means missing CD or CI metric):')
print(brickts_wide[['perf_score', 'time_score', 'mem_score']].isna().sum())
print('\nScore distribution summary:')
print(brickts_wide[['perf_score', 'time_score', 'mem_score']].describe().round(3))
brickts_wide.head()


In [ ]:
# Sanity: score 분포 히스토그램 (단위 차별)
#  - perf_score:  mse 기반 (의미상 % 단위). 표시는 *100 → -100~+100% 로 변환
#  - time_score / mem_score: efficiency 기반 (배수 단위). -1~+1 그대로 + big/small 직관 caption
score_specs = [
    ('perf_score', lambda v: v * 100, 'Performance diff (%)  (+CI better, -CD better)'),
    ('time_score', lambda v: v,        'Time efficiency      (-1~+1, +CI faster)'),
    ('mem_score',  lambda v: v,        'Memory efficiency    (-1~+1, +CI lighter, num_params 기반)'),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, scaler, label) in zip(axes, score_specs):
    vals = scaler(brickts_wide[col].dropna())
    sns.histplot(vals, bins=30, ax=ax, color='steelblue')
    ax.axvline(0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlabel(label)
    ax.set_title(col)
plt.tight_layout(); plt.show()

print('Mean score per dataset (positive => CI typically better):')
ds_summary = (brickts_wide.groupby('dataset')[['perf_score', 'time_score', 'mem_score']]
              .mean().round(3))
ds_summary


## Section 3 — 외부 벤치마크 결과 로딩 + (arch, scope, level) 매핑

외부 20+ 모델 (Transformer, TimesNet, DLinear, …) 의 결과를 archieve `MODEL_GROUPS` 매핑을 그대로 사용하여 (arch, scope, level) bucket 으로 분류. 메타러너의 holdout test set 으로 사용한다.

**Per-model 짝짓기 → bucket median**:
- 각 외부 모델 m 에 대해 (mse_CD, mse_CI) 쌍 → score_m 계산
- 같은 (arch, scope, level) bucket 에 매핑된 모델들의 median 으로 bucket score 산출


In [ ]:
MODEL_GROUPS = {
    'architecture': {
        'mlp':         ['linear', 'dlinear', 'tsmixer', 'timemixer', 'lightts', 'tide', 'duet', 'wpmixer'],
        'cnn':         ['tcn', 'scinet', 'timesnet', 'micn'],
        'rnn':         ['rnn', 'segrnn', 'dssrnn', 'ssrnn'],
        'transformer': ['transformer', 'pyraformer', 'autoformer', 'informer', 'itransformer', 'patchtst'],
    },
    'level': {
        'direct':        ['tcn', 'linear', 'rnn', 'transformer', 'segrnn', 'lightts', 'tide', 'ssrnn', 'informer'],
        'spectral':      ['timemixer', 'autoformer', 'timesnet', 'duet', 'wpmixer'],
        'decomposition': ['dlinear', 'tsmixer', 'scinet', 'pyraformer', 'dssrnn', 'micn'],
    },
    'scope': {
        'global':       ['linear', 'dlinear', 'rnn', 'tsmixer', 'transformer', 'dssrnn', 'lightts', 'tide', 'ssrnn'],
        'sparse':       ['autoformer', 'informer', 'duet'],
        'hierarchical': ['scinet', 'pyraformer', 'timemixer', 'timesnet', 'micn', 'wpmixer'],
        'local':        ['tcn', 'segrnn'],
    },
}


def classify_model(model: str, dim: str) -> Optional[str]:
    ml = model.lower()
    for cat, members in MODEL_GROUPS[dim].items():
        if ml in members:
            return cat
    return None


def load_benchmark_ot_results(results_dir: str, ot_name_by_dataset: dict) -> pd.DataFrame:
    rows = []
    for fp in glob.glob(f'{results_dir}/**/*_results.json', recursive=True):
        path = Path(fp)
        if 'BrickTS' in path.parts or path.name.startswith('BrickTS'):
            continue
        parts = path.parts
        try:
            ri = parts.index('results')
        except ValueError:
            continue
        if ri + 3 >= len(parts):
            continue
        ds_folder, hz = parts[ri + 1], parts[ri + 2]
        ds = _folder_to_name(ds_folder)
        m = re.match(r'^(?P<model>.+)_(?P<mode>CD|CI)_results\.json$', path.name)
        if not m:
            continue
        model, mode = m.group('model'), m.group('mode')
        try:
            seq_len, pred_len = map(int, hz.split('_'))
        except Exception:
            seq_len = pred_len = -1
        with open(fp) as f:
            data = json.load(f)
        ot_name = ot_name_by_dataset.get(ds, 'OT')
        mse, mae, n_params, t = _ot_metrics_from_payload(data, mode, ot_name)
        rows.append({
            'dataset': ds, 'model': model, 'strategy': mode,
            'arch':  classify_model(model, 'architecture'),
            'scope': classify_model(model, 'scope'),
            'level': classify_model(model, 'level'),
            'seq_len': seq_len, 'pred_len': pred_len,
            'mse': mse, 'mae': mae, 'num_params': n_params, 'train_time': t,
        })
    return pd.DataFrame(rows)


benchmark_long = pd.DataFrame()
for cand in ['../results', 'results', './results']:
    df = load_benchmark_ot_results(cand, ot_map)
    if len(df):
        benchmark_long = df
        print(f'Using benchmark results_dir = {cand}')
        break

benchmark_long.to_csv(OUT / 'benchmark_long.csv', index=False)
print('\nBenchmark rows:', len(benchmark_long))
unmapped = benchmark_long[benchmark_long[['arch', 'scope', 'level']].isna().any(axis=1)]
print(f'Unmapped models (skipped from bucket aggregation): {sorted(unmapped["model"].str.lower().unique())}')
benchmark_long.head()


In [ ]:
# Per-model wide (CD/CI 둘 다 있는 모델만 유지) + score 계산
bench_wide_per_model = benchmark_long.pivot_table(
    index=['dataset', 'model', 'arch', 'scope', 'level'],
    columns='strategy',
    values=['mse', 'train_time', 'num_params'],
).reset_index()
bench_wide_per_model.columns = [f'{a}_{b}' if b else a for a, b in bench_wide_per_model.columns]
# pivot_table 누락 컬럼 안전장치
for col in REQUIRED_WIDE_COLS:
    if col not in bench_wide_per_model.columns:
        bench_wide_per_model[col] = np.nan
bench_wide_per_model = bench_wide_per_model.dropna(subset=['mse_CD', 'mse_CI'])
bench_wide_per_model['perf_score'] = compute_perf_score(bench_wide_per_model['mse_CD'],        bench_wide_per_model['mse_CI'])
bench_wide_per_model['time_score'] = compute_efficiency_score(bench_wide_per_model['train_time_CD'], bench_wide_per_model['train_time_CI'])
bench_wide_per_model['mem_score']  = compute_efficiency_score(bench_wide_per_model['num_params_CD'], bench_wide_per_model['num_params_CI'])
bench_wide_per_model.to_csv(OUT / 'benchmark_wide_per_model.csv', index=False)

print(f'bench_wide_per_model shape: {bench_wide_per_model.shape}')
print('\nModels per dataset (CD/CI pair available):')
print(bench_wide_per_model.groupby('dataset')['model'].nunique().sort_index())

# Bucket median (mapped only)
bench_wide_bucket = (bench_wide_per_model
    .dropna(subset=['arch', 'scope', 'level'])
    .groupby(['dataset', 'arch', 'scope', 'level'])
    [['perf_score', 'time_score', 'mem_score']].median().reset_index())
bench_wide_bucket.to_csv(OUT / 'benchmark_wide_bucket.csv', index=False)

print(f'\nbench_wide_bucket shape: {bench_wide_bucket.shape}')
print('Buckets per dataset:')
print(bench_wide_bucket.groupby('dataset').size().sort_index())
bench_wide_bucket.head()


## Section 4 — USER CONFIG

학습 타깃, 모델, feature subset, 분류 임계값을 한 곳에서 설정.


In [ ]:
# ===== USER CONFIG =====
import random
# Milano 지역을 7:1:2 (train:val:test) 로 분할.
#  - train : 학습
#  - val   : early stopping (validation 성능 최적화) — inference 에는 사용 안 함
#  - test  : NB3 §3 inference holdout (NB2 에서 사용 안 함)
MILANO_POOL_N = None              # 사용할 Milano 총 개수 제한 (None=전부, int=랜덤 N개만)
SPLIT_RATIOS  = (0.7, 0.1, 0.2)   # (train, val, test)
SPLIT_SEED    = 42                # 재현용 시드

_milano_pat = re.compile(r'^Milano-\d+$')
# results_BrickTS(brickts_long) 와 NB1 메타피처(meta_full) 양쪽에 존재하는 Milano 만 후보
_all_milano = sorted([d for d in brickts_long['dataset'].unique()
                      if _milano_pat.match(str(d)) and d in meta_full.index],
                     key=lambda s: int(s.split('-')[1]))
_rng = random.Random(SPLIT_SEED)
if MILANO_POOL_N is not None and MILANO_POOL_N < len(_all_milano):
    _all_milano = sorted(_rng.sample(_all_milano, MILANO_POOL_N), key=lambda s: int(s.split('-')[1]))

_shuf = _all_milano[:]; _rng.shuffle(_shuf)
_n = len(_shuf)
_n_tr = round(_n * SPLIT_RATIOS[0])
_n_va = round(_n * SPLIT_RATIOS[1])
_keyf = lambda s: int(s.split('-')[1])
TRAIN_DATASETS = sorted(_shuf[:_n_tr], key=_keyf)
VAL_DATASETS   = sorted(_shuf[_n_tr:_n_tr + _n_va], key=_keyf)
TEST_DATASETS  = sorted(_shuf[_n_tr + _n_va:], key=_keyf)   # NB3 inference holdout
print(f'All Milano (results & meta) : {_n}  | split {SPLIT_RATIOS}')
print(f'TRAIN ({len(TRAIN_DATASETS)}) : {TRAIN_DATASETS}')
print(f'VAL   ({len(VAL_DATASETS)}) : {VAL_DATASETS}')
print(f'TEST/inference ({len(TEST_DATASETS)}) : {TEST_DATASETS}')

# Feature subset (None: 전체 사용)
SELECTED_FEATURES: Optional[list] = None
SELECTED_GROUPS:   Optional[list] = None        # 예: ['A', 'B', 'D']
SELECTED_PATTERN:  Optional[str]  = None
EXCLUDED_FEATURES: list           = []
EXCLUDED_PATTERN:  Optional[str]  = None

# Targets / model / classification
TARGET_COLS      = ['perf_score', 'time_score', 'mem_score']   # mem_score = num_params 기반
PRIMARY_TARGET   = 'perf_score'
MODEL_NAME       = 'XGBoost'       # 사용할 단일 모델 직접 선택: 'LightGBM' | 'XGBoost'
MODEL_CANDIDATES = [MODEL_NAME]
USE_GPU          = True            # LightGBM/XGBoost GPU 학습 사용 여부
GPU_PARAMS       = {               # GPU 환경/버전에 맞게 조정 가능
    'LightGBM': {'device': 'gpu'},
    'XGBoost':  {'device': 'cuda', 'tree_method': 'hist'},
}
# val early stopping 으로 best iteration 자동 선택 → n_estimators 는 상한으로 크게.
EARLY_STOPPING_ROUNDS = 50
DEFAULT_HP = {
    'LightGBM': {'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.05,
                 'num_leaves': 8, 'min_child_samples': 10},
    'XGBoost':  {'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.05,
                 'min_child_weight': 5, 'reg_alpha': 0.1, 'reg_lambda': 1.0},
}
BEST_METRIC = 'auc'                # 참고 지표 (PRIMARY_TARGET 의 CD/CI 분리 능력)
# =======================

print(f'MODEL_NAME = {MODEL_NAME}  |  USE_GPU = {USE_GPU}  |  ES rounds = {EARLY_STOPPING_ROUNDS}')
print(f'TARGET_COLS = {TARGET_COLS}  (primary = {PRIMARY_TARGET})  |  BEST_METRIC = {BEST_METRIC}')


## Section 5 — Feature subset + Feature-Target 매트릭스

NB1 메타피처를 (dataset, arch, scope, level) 매트릭스에 join + (arch, scope, level) one-hot. strategy 는 더 이상 입력이 아니다 (타깃이 score 로 흡수됨).


In [ ]:
def apply_feature_selection(meta_df: pd.DataFrame, fg: pd.DataFrame) -> list:
    feats = list(meta_df.columns)
    grp_map = dict(zip(fg['feature'], fg['group']))
    if SELECTED_FEATURES is not None:
        feats = [f for f in SELECTED_FEATURES if f in feats]
    elif SELECTED_GROUPS is not None:
        feats = [f for f in feats if grp_map.get(f) in SELECTED_GROUPS]
    elif SELECTED_PATTERN is not None:
        rx = re.compile(SELECTED_PATTERN)
        feats = [f for f in feats if rx.search(f)]
    feats = [f for f in feats if f not in set(EXCLUDED_FEATURES)]
    if EXCLUDED_PATTERN is not None:
        rx2 = re.compile(EXCLUDED_PATTERN)
        feats = [f for f in feats if not rx2.search(f)]
    return feats


selected_feats = apply_feature_selection(meta_full, feature_groups)
print(f'Selected features: {len(selected_feats)} / {meta_full.shape[1]}')
print('\nGroup distribution:')
print(feature_groups.set_index('feature').loc[selected_feats, 'group'].value_counts())


In [ ]:
def build_feature_target_table(wide: pd.DataFrame, meta_df: pd.DataFrame,
                                feat_cols: list) -> pd.DataFrame:
    """Join meta features + (arch, scope, level) one-hot. strategy 는 입력에서 제외."""
    df = wide.merge(meta_df[feat_cols].reset_index(), on='dataset', how='left')
    for c in feat_cols:
        df[c] = df[c].fillna(df[c].median())
    onehot = pd.concat([
        pd.get_dummies(df['arch'],  prefix='arch'),
        pd.get_dummies(df['scope'], prefix='scope'),
        pd.get_dummies(df['level'], prefix='level'),
    ], axis=1)
    return pd.concat([df, onehot], axis=1)


FT = build_feature_target_table(brickts_wide, meta_full, selected_feats)
ONEHOT_COLS = [c for c in FT.columns if c.startswith(('arch_', 'scope_', 'level_'))]
X_COLS = list(selected_feats) + ONEHOT_COLS

# perf_score (PRIMARY_TARGET) 는 필수 — 정의 불가(NaN) 시 제외
n_before = len(FT)
FT = FT.dropna(subset=[PRIMARY_TARGET]).reset_index(drop=True)

# cost score (time/mem) 결측은 동률(0) 로 대체.
#   일부 데이터셋의 num_params / train_time 이 누락돼도 그 데이터셋이 통째로 빠지지 않도록.
cost_targets = [t for t in TARGET_COLS if t != PRIMARY_TARGET]
for t in cost_targets:
    n_nan = int(FT[t].isna().sum())
    if n_nan:
        print(f'  [{t}] {n_nan} rows NaN → 0(동률)으로 대체 (cost 메트릭 누락)')
    FT[t] = FT[t].fillna(0.0)

n_after_nan = len(FT)
# 학습/검증(train+val) 지역만 사용 (test 지역은 NB3 inference 전용)
_trainval = TRAIN_DATASETS + VAL_DATASETS
FT = FT[FT['dataset'].isin(_trainval)].reset_index(drop=True)
print(f'FT shape: {FT.shape}  |  X_COLS: {len(X_COLS)}')
print(f'  dropped {n_before - n_after_nan} rows with NaN {PRIMARY_TARGET}, '
      f'then {n_after_nan - len(FT)} rows not in train+val')
print(f'  datasets in FT: {sorted(FT["dataset"].unique())}')
missing = sorted(set(_trainval) - set(FT['dataset'].unique()))
if missing:
    print(f'  [!] train+val 중 FT 에 없음: {missing}')
    for ds in missing:
        s = brickts_wide[brickts_wide['dataset'] == ds]
        print(f'        {ds}: rows={len(s)}, perf_score NaN={int(s["perf_score"].isna().sum()) if len(s) else "-"}')
FT[['dataset', 'arch', 'scope', 'level'] + TARGET_COLS].head()


## Section 6 — 학습 (train) + validation early stopping

Milano 지역 7:1:2 분할 중 **train 으로 학습, val 로 early stopping** (best iteration 자동 선택 → validation 성능 최적화). test 지역은 NB3 §3 inference 전용.

target 별 단일 모델을 각각 val 로 early stopping 후 `MultiOutputRegressor` 형태로 묶는다. 아래 §7 표·그림은 모두 **validation(held-out) 기준**. `USE_GPU=True` 면 `GPU_PARAMS` 적용.

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor


def get_base_model(name: str, **params):
    if USE_GPU:
        params = {**GPU_PARAMS.get(name, {}), **params}   # USER HP 가 GPU 기본값 override
    if name == 'XGBoost':
        return xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, **params)
    return lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1, **params)


def fit_with_early_stopping(name, params, X_tr, Y_tr, X_va, Y_va, target_cols, es_rounds):
    """target 별 단일 모델을 validation early stopping 으로 학습 →
    sklearn MultiOutputRegressor 형태로 묶어 반환 (pickle 호환)."""
    use_es = X_va is not None and len(X_va) > 0
    estimators, best_iters = [], {}
    for tgt in target_cols:
        if name == 'XGBoost':
            if use_es:
                est = get_base_model(name, early_stopping_rounds=es_rounds, **params)
                est.fit(X_tr, Y_tr[tgt], eval_set=[(X_va, Y_va[tgt])], verbose=False)
            else:
                est = get_base_model(name, **params); est.fit(X_tr, Y_tr[tgt])
            best_iters[tgt] = int(getattr(est, 'best_iteration', -1) or -1)
        else:  # LightGBM
            est = get_base_model(name, **params)
            if use_es:
                est.fit(X_tr, Y_tr[tgt], eval_set=[(X_va, Y_va[tgt])],
                        callbacks=[lgb.early_stopping(es_rounds, verbose=False),
                                   lgb.log_evaluation(0)])
            else:
                est.fit(X_tr, Y_tr[tgt])
            best_iters[tgt] = int(getattr(est, 'best_iteration_', -1) or -1)
        estimators.append(est)
    mor = MultiOutputRegressor(get_base_model(name, **params))   # 빈 템플릿
    mor.estimators_ = estimators
    mor.n_features_in_ = X_tr.shape[1]
    return mor, best_iters


### 학습 (train) + early stopping (val)

train 지역으로 학습하되 **val 지역으로 early stopping** (best iteration 자동 선택). 표/그림은 **validation 기준** 평가.
target 별 `mse / mae / spearman / sign_acc / accuracy / f1 / auc`. 2-class 라벨: `score < 0 → 'CD'`, `score >= 0 → 'CI'` (sign 경계).

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, f1_score, accuracy_score, mean_squared_error, roc_auc_score


def to_two_class(y: np.ndarray) -> np.ndarray:
    """score < 0 -> 'CD',  score >= 0 -> 'CI'  (sign 경계)."""
    return np.where(y >= 0, 'CI', 'CD')


def per_target_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[m], y_pred[m]
    if len(y_true) < 2:
        nan = float('nan')
        return {'mse': nan, 'mae': nan, 'spearman': nan,
                'sign_acc': nan, 'accuracy': nan, 'f1': nan, 'auc': nan}
    sp, _ = spearmanr(y_true, y_pred)
    cls_true = to_two_class(y_true)
    cls_pred = to_two_class(y_pred)
    labels = ['CD', 'CI']
    yt_bin = (y_true >= 0).astype(int)   # CI=1, CD=0
    try:
        auc = float(roc_auc_score(yt_bin, y_pred)) if len(np.unique(yt_bin)) == 2 else float('nan')
    except ValueError:
        auc = float('nan')
    return {
        'mse':       float(mean_squared_error(y_true, y_pred)),
        'mae':       float(mean_absolute_error(y_true, y_pred)),
        'spearman':  float(sp) if not np.isnan(sp) else 0.0,
        'sign_acc':  float(np.mean(np.sign(y_true) == np.sign(y_pred))),
        'accuracy':  float(accuracy_score(cls_true, cls_pred)),
        'f1':        float(f1_score(cls_true, cls_pred, average='macro',
                                     labels=labels, zero_division=0)),
        'auc':       auc,
    }


# ── train/val 분할 (지역 단위) + early stopping 학습 ──────────
ft_tr = FT[FT['dataset'].isin(TRAIN_DATASETS)].reset_index(drop=True)
ft_va = FT[FT['dataset'].isin(VAL_DATASETS)].reset_index(drop=True)
X_tr, Y_tr = ft_tr[X_COLS].astype(float), ft_tr[TARGET_COLS]
X_va, Y_va = ft_va[X_COLS].astype(float), ft_va[TARGET_COLS]
print(f'train rows {len(ft_tr)} ({ft_tr["dataset"].nunique()} ds)  |  val rows {len(ft_va)} ({ft_va["dataset"].nunique()} ds)')
if len(ft_va) == 0:
    print('[!] VAL 비어있음 — early stopping 없이 train 으로만 학습')

MODEL_USED   = MODEL_CANDIDATES[0]
MODEL_PARAMS = DEFAULT_HP[MODEL_USED]
full_model, best_iters = fit_with_early_stopping(
    MODEL_USED, MODEL_PARAMS, X_tr, Y_tr, X_va, Y_va, TARGET_COLS, EARLY_STOPPING_ROUNDS)
print(f'Trained {MODEL_USED} (GPU={USE_GPU})  |  best_iteration per target: {best_iters}')

# validation 예측 + 메트릭 (val 비면 train 으로 대체 표시)
eval_ft = ft_va if len(ft_va) else ft_tr
X_eval = eval_ft[X_COLS].astype(float)
Y_pred_eval = np.asarray(full_model.predict(X_eval), dtype=float)
preds_df = eval_ft[['dataset', 'arch', 'scope', 'level'] + TARGET_COLS].copy()
preds_df['model'] = MODEL_USED
for j, tgt in enumerate(TARGET_COLS):
    preds_df[f'pred_{tgt}'] = Y_pred_eval[:, j]

metric_rows = []
for ds in preds_df['dataset'].unique():
    sub = preds_df[preds_df['dataset'] == ds]
    for tgt in TARGET_COLS:
        metric_rows.append({'model': MODEL_USED, 'target': tgt, 'held_out': ds,
                            **per_target_metrics(sub[tgt].values, sub[f'pred_{tgt}'].values)})
metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(OUT / 'ml_metrics_per_target.csv', index=False)
preds_df.to_csv(OUT / 'ml_predictions.csv', index=False)

print('\nPer (model, target) VALIDATION 평균:')
metrics_df.groupby(['model', 'target'])[['mse', 'mae', 'spearman', 'sign_acc', 'accuracy', 'f1', 'auc']].mean().round(3)


## Section 7 — 평가 시각화

- 모델별 데이터셋별 sign accuracy (per target)
- 모델별 PRIMARY_TARGET 의 3-class confusion matrix (LODO pooled)


In [ ]:
# 모델별 validation sign_acc per dataset (각 모델 한 row)
fig, axes = plt.subplots(len(MODEL_CANDIDATES), 1,
                          figsize=(13, 4.5 * len(MODEL_CANDIDATES)), squeeze=False)
for ax, name in zip(axes[:, 0], MODEL_CANDIDATES):
    sub = metrics_df[metrics_df['model'] == name]
    sns.barplot(data=sub, x='held_out', y='sign_acc', hue='target', ax=ax)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.6)
    ax.set_title(f'validation sign_acc per dataset  |  model = {name}')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=20)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

LABELS = ['CD', 'CI']

# 모델별 PRIMARY_TARGET 의 validation 2-class(CD/CI) confusion matrix
fig, axes = plt.subplots(1, len(MODEL_CANDIDATES),
                          figsize=(5 * len(MODEL_CANDIDATES), 4.5), squeeze=False)
for ax, name in zip(axes[0], MODEL_CANDIDATES):
    sub = preds_df[preds_df['model'] == name]
    m = np.isfinite(sub[PRIMARY_TARGET]) & np.isfinite(sub[f'pred_{PRIMARY_TARGET}'])
    yt = to_two_class(sub.loc[m, PRIMARY_TARGET].values)
    yp = to_two_class(sub.loc[m, f'pred_{PRIMARY_TARGET}'].values)
    cm = confusion_matrix(yt, yp, labels=LABELS)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    f1 = f1_score(yt, yp, average='macro', labels=LABELS, zero_division=0)
    acc = accuracy_score(yt, yp)
    ax.set_title(f'{name}  |  {PRIMARY_TARGET}\nF1 = {f1:.3f}  acc = {acc:.3f}')
plt.tight_layout(); plt.show()

In [ ]:
# AUC(ROC) 시각화 — validation, PRIMARY_TARGET(CD vs CI)
# 예측 점수(pred_perf_score)로 실제 CI(+)/CD(-) 를 얼마나 잘 분리하는지.
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(6.2, 5.6))
for name in MODEL_CANDIDATES:
    sub = preds_df[preds_df['model'] == name]
    mask = np.isfinite(sub[PRIMARY_TARGET]) & np.isfinite(sub[f'pred_{PRIMARY_TARGET}'])
    yt = (sub.loc[mask, PRIMARY_TARGET].values >= 0).astype(int)   # CI=1, CD=0
    ys = sub.loc[mask, f'pred_{PRIMARY_TARGET}'].values
    if len(np.unique(yt)) < 2:
        print(f'[{name}] 한 클래스만 존재 — ROC 생략')
        continue
    fpr, tpr, _ = roc_curve(yt, ys)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc_score(yt, ys):.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='chance')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate (CI=positive)')
ax.set_title(f'Validation ROC — {PRIMARY_TARGET} (CD vs CI)')
ax.legend(loc='lower right'); plt.tight_layout(); plt.show()

## Section 8 — Target 별 feature importance

단일 고정 모델(§6 에서 학습한 `full_model`)의 target 별 native gain importance 추출.

In [ ]:
def native_importance(estimator, name: str, feats: list) -> np.ndarray:
    if name == 'LightGBM':
        return np.asarray(estimator.booster_.feature_importance(importance_type='gain'), dtype=float)
    gains = estimator.get_booster().get_score(importance_type='gain')
    return np.asarray([gains.get(f, gains.get(f'f{i}', 0.0))
                        for i, f in enumerate(feats)], dtype=float)


# 단일 고정 모델 (LODO/모델선택 불필요) — §6 에서 학습한 full_model 재사용
BEST_MODEL_NAME = MODEL_USED
BEST_PARAMS     = MODEL_PARAMS
print(f'Model: {BEST_MODEL_NAME}  |  HP: {BEST_PARAMS}  |  BEST_METRIC(ref) = {BEST_METRIC}')

# Target 별 importance
imp_rows = []
for j, tgt in enumerate(TARGET_COLS):
    est = full_model.estimators_[j]
    vals = native_importance(est, BEST_MODEL_NAME, X_COLS)
    rank = pd.Series(vals).rank(ascending=False, method='min').astype(int).values
    for f, v, r in zip(X_COLS, vals, rank):
        imp_rows.append({'model': BEST_MODEL_NAME, 'target': tgt,
                          'feature': f, 'importance': float(v), 'rank': int(r)})
fi_tree = pd.DataFrame(imp_rows)
fi_tree.to_csv(OUT / 'feature_importance_tree.csv', index=False)


fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, tgt in zip(axes, TARGET_COLS):
    sub = fi_tree[fi_tree['target'] == tgt].nlargest(15, 'importance')
    sns.barplot(data=sub, y='feature', x='importance', ax=ax, color='steelblue')
    ax.set_title(f'Top-15 importance — {tgt}')
plt.suptitle(f'{BEST_MODEL_NAME} native gain importance per target', y=1.01)
plt.tight_layout(); plt.show()


## Section 9 — Best 모델 bundle 저장

§8 에서 재학습된 `full_model` 을 metadata 와 함께 `best_model.pkl` 에 저장. NB3 §3 에서 신규 지역 추론에 사용.

In [ ]:
bundle = {
    'model': full_model,
    'model_name': BEST_MODEL_NAME,
    'model_candidates': MODEL_CANDIDATES,
    'targets': TARGET_COLS,
    'primary_target': PRIMARY_TARGET,
    'hyperparams': BEST_PARAMS,
    'feature_list': X_COLS,
    'selected_meta_features': selected_feats,
    'onehot_cols': ONEHOT_COLS,
    'train_datasets': TRAIN_DATASETS,
    'val_datasets': VAL_DATASETS,
    'test_datasets': TEST_DATASETS,
}
with open(OUT / 'best_model.pkl', 'wb') as f:
    pickle.dump(bundle, f)

pd.DataFrame([{
    'model_name': BEST_MODEL_NAME,
    'model_candidates': json.dumps(MODEL_CANDIDATES),
    'targets': json.dumps(TARGET_COLS),
    'primary_target': PRIMARY_TARGET,
    'hyperparameters_json': json.dumps(BEST_PARAMS),
    'train_datasets_json': json.dumps(TRAIN_DATASETS),
    'val_datasets_json': json.dumps(VAL_DATASETS),
    'test_datasets_json': json.dumps(TEST_DATASETS),
    'feature_list_json': json.dumps(X_COLS),
    'meta_features_json': json.dumps(selected_feats),
}]).to_csv(OUT / 'best_model_config.csv', index=False)
print(f'Saved best_model.pkl  ({BEST_MODEL_NAME})  +  best_model_config.csv')

## Section 10 — BrickTS vs 외부 벤치마크 raw 결과 비교 (sanity check)

메타러너 holdout test 가 아니다 — 외부 벤치마크 결과는 동일 데이터셋·동일 (arch, scope, level) 축에서 측정된 것이므로 메타러너 일반화 평가가 될 수 없음. 대신, 같은 (dataset, arch, scope, level) bucket 에서 **BrickTS 와 외부 모델들이 동일한 CD/CI 우열 패턴을 보이는지** 확인하는 raw 비교만 수행.

비교 컬럼: `brickts_*` (BrickTS wide 점수), `bench_*` (외부 모델 bucket median), `diff_*` (bench − brickts). 3 점수 모두 (perf/time/params).

메타러너 일반화 검증은 **NB3 §3 신규 지역 (Milano-8815) 추론** 으로 분리.

In [ ]:
join_keys = ['dataset', 'arch', 'scope', 'level']

# bucket 당 모델 수 (외부)
bucket_n = (bench_wide_per_model.dropna(subset=['arch', 'scope', 'level'])
            .groupby(join_keys).size().rename('n_models').reset_index())

# BrickTS wide 와 외부 bucket median 을 inner join
score_cols    = ['perf_score', 'time_score', 'mem_score']
brickts_score = {'perf_score': 'brickts_perf', 'time_score': 'brickts_time', 'mem_score': 'brickts_mem'}
bench_score   = {'perf_score': 'bench_perf',   'time_score': 'bench_time',   'mem_score': 'bench_mem'}

brickts_bench = (
    brickts_wide[join_keys + score_cols].rename(columns=brickts_score)
    .merge(
        bench_wide_bucket[join_keys + score_cols].rename(columns=bench_score),
        on=join_keys, how='inner',
    )
    .merge(bucket_n, on=join_keys, how='left')
)
for sc in ['perf', 'time', 'mem']:
    brickts_bench[f'diff_{sc}'] = brickts_bench[f'bench_{sc}'] - brickts_bench[f'brickts_{sc}']

out_cols = join_keys + ['n_models',
                         'brickts_perf', 'bench_perf', 'diff_perf',
                         'brickts_time', 'bench_time', 'diff_time',
                         'brickts_mem',  'bench_mem',  'diff_mem']
brickts_bench[out_cols].to_csv(OUT / 'brickts_vs_benchmark.csv', index=False)
print(f'Compared buckets: {len(brickts_bench)}')
print('Per-dataset bucket coverage:')
print(brickts_bench.groupby('dataset').size().sort_index())
brickts_bench[out_cols].head()


In [ ]:
from scipy.stats import pearsonr, spearmanr

# 세 점수별 scatter + Pearson r + Spearman ρ
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, sc in zip(axes, ['perf', 'time', 'mem']):
    sub = brickts_bench[[f'brickts_{sc}', f'bench_{sc}']].dropna()
    if len(sub) > 2:
        rp, _ = pearsonr(sub[f'brickts_{sc}'], sub[f'bench_{sc}'])
        rs, _ = spearmanr(sub[f'brickts_{sc}'], sub[f'bench_{sc}'])
        title = f'{sc}_score   r={rp:.3f}, ρ={rs:.3f}  (n={len(sub)})'
    else:
        title = f'{sc}_score  (insufficient data)'
    ax.scatter(sub[f'brickts_{sc}'], sub[f'bench_{sc}'], alpha=0.6, s=40, color='steelblue')
    ax.plot([-1, 1], [-1, 1], 'k--', alpha=0.4)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
    ax.axvline(0, color='gray', linestyle=':', alpha=0.4)
    ax.set_xlabel(f'BrickTS {sc}_score')
    ax.set_ylabel(f'Benchmark {sc}_score (bucket median)')
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
    ax.set_title(title)
plt.suptitle('BrickTS vs Benchmark (raw score agreement)', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# §10 추가: BrickTS vs 외부 벤치마크 — perf_score 3-class confusion matrix
# (메타러너 평가가 아닌, raw 결과의 CD/CI 분류 일치도)
m = pd.notna(brickts_bench['brickts_perf']) & pd.notna(brickts_bench['bench_perf'])
yt = to_two_class(brickts_bench.loc[m, 'brickts_perf'].values)
yp = to_two_class(brickts_bench.loc[m, 'bench_perf'].values)
cm_bb = confusion_matrix(yt, yp, labels=LABELS)
f1_bb = f1_score(yt, yp, average='macro', labels=LABELS, zero_division=0)
acc_bb = accuracy_score(yt, yp)

fig, ax = plt.subplots(figsize=(5, 4.5))
sns.heatmap(cm_bb, annot=True, fmt='d', cmap='Purples',
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel('Benchmark (bucket median)')
ax.set_ylabel('BrickTS')
ax.set_title(f'§10 BrickTS vs Benchmark — perf_score 2-class(CD/CI)\n'
              f'F1 = {f1_bb:.3f}, acc = {acc_bb:.3f}, n = {len(yt)}')
plt.tight_layout(); plt.show()
